# 06 — Topological Clustering: ToMATo vs the Field

**Thesis:** topology-aware clustering separates rings, spirals, and voids that
convex-cluster methods can't touch — but pays a price on high-dimensional noise.

**Method:** ToMATo (Topological Mode Analysis Tool) from `gudhi`.  
Builds a k-NN density graph, then uses persistent homology to decide when to merge
density modes. The single extra hyper-parameter is `k` (neighbour count).

**Datasets:**
| Dataset | n | d | k | Notes |
|---------|---|---|---|-------|
| circles | 300 | 2 | 2 | synthetic concentric rings — topology showcase |
| penguins | 333 | 4 | 3 | real, Gaussian-ish blobs |
| wine | 178 | 13 | 3 | moderate-d, convex clusters |
| digits 0–4 | 901 | 64 | 5 | high-d, topological method suffers |


In [ ]:
from __future__ import annotations

import sys
import time
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from sklearn.metrics import adjusted_rand_score

from src.algorithms import (
    run_agglomerative_ward,
    run_agglomerative_complete,
    run_gmm,
    run_hdbscan,
    run_kmeans,
    run_tomato,
)
from src.datasets import load_circles, load_digits_subset, load_penguins, load_wine_data

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


In [ ]:
DATASETS = {
    "circles": load_circles(),
    "penguins": load_penguins(),
    "wine": load_wine_data(),
    "digits (0-4)": load_digits_subset(),
}

for name, (x, y) in DATASETS.items():
    n_cls = len(np.unique(y))
    print(f"{name:15s}  n={len(x):4d}  d={x.shape[1]:2d}  k={n_cls}")


In [ ]:
def benchmark(x: np.ndarray, y: np.ndarray) -> dict[str, dict]:
    n_clusters = len(np.unique(y))
    algos = {
        "KMeans":   lambda: run_kmeans(x, n_clusters),
        "Ward":     lambda: run_agglomerative_ward(x, n_clusters),
        "Complete": lambda: run_agglomerative_complete(x, n_clusters),
        "HDBSCAN":  lambda: run_hdbscan(x, min_cluster_size=5),
        "GMM":      lambda: run_gmm(x, n_clusters),
        "ToMATo":   lambda: run_tomato(x, n_clusters, k=10),
    }
    out = {}
    for name, fn in algos.items():
        t0 = time.perf_counter()
        labels = fn()
        elapsed = time.perf_counter() - t0
        mask = labels >= 0
        ari = adjusted_rand_score(y[mask], labels[mask]) if mask.sum() > 0 else float("nan")
        out[name] = {"ari": ari, "noise_pct": (~mask).mean() * 100, "time_s": elapsed}
    return out

results = {name: benchmark(x, y) for name, (x, y) in DATASETS.items()}
print("Benchmark complete.")


In [ ]:
algo_names = list(next(iter(results.values())).keys())
ds_names = list(results.keys())

ari_matrix = np.array(
    [[results[ds][algo]["ari"] for algo in algo_names] for ds in ds_names]
)

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(ari_matrix, vmin=-0.1, vmax=1.0, cmap="RdYlGn", aspect="auto")
fig.colorbar(im, ax=ax, label="ARI (higher = better)")

ax.set_xticks(range(len(algo_names)))
ax.set_xticklabels(algo_names, rotation=30, ha="right")
ax.set_yticks(range(len(ds_names)))
ax.set_yticklabels(ds_names)
ax.set_title("Adjusted Rand Index — ToMATo enters the arena", fontsize=13)

for i in range(len(ds_names)):
    for j in range(len(algo_names)):
        v = ari_matrix[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="black" if 0.2 < v < 0.8 else "white")

plt.tight_layout()
plt.savefig("/tmp/06_ari_heatmap.png")
plt.show()


In [ ]:
x_c, y_c = DATASETS["circles"]
n_cls = len(np.unique(y_c))

algo_scatter = {
    "Ground truth": y_c,
    "KMeans":   run_kmeans(x_c, n_cls),
    "Ward":     run_agglomerative_ward(x_c, n_cls),
    "HDBSCAN":  run_hdbscan(x_c, min_cluster_size=5),
    "GMM":      run_gmm(x_c, n_cls),
    "ToMATo":   run_tomato(x_c, n_cls, k=10),
}

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
fig.suptitle("Two concentric rings — who survives?", fontsize=13)

for ax, (title, labels) in zip(axes.flat, algo_scatter.items()):
    mask = labels >= 0
    ari = adjusted_rand_score(y_c[mask], labels[mask]) if title != "Ground truth" else 1.0
    ax.scatter(x_c[:, 0], x_c[:, 1], c=labels, cmap="tab10", s=14, linewidths=0)
    noise_label = f"  ({(~mask).sum()} noise)" if (~mask).any() else ""
    suffix = f"  ARI={ari:.3f}" if title != "Ground truth" else ""
    ax.set_title(f"{title}{suffix}{noise_label}", fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")

plt.tight_layout()
plt.savefig("/tmp/06_circles_scatter.png")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(ds_names), figsize=(13, 3.5), sharey=False)
fig.suptitle("Runtime (seconds)", fontsize=12)

for ax, ds_name in zip(axes, ds_names):
    times = [results[ds_name][a]["time_s"] for a in algo_names]
    colors = ["#e07b54" if a == "ToMATo" else "#7bafd4" for a in algo_names]
    ax.barh(algo_names, times, color=colors)
    ax.set_title(ds_name, fontsize=9)
    ax.set_xlabel("s")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

plt.tight_layout()
plt.savefig("/tmp/06_runtime.png")
plt.show()


## Key Findings

| Dataset | ToMATo ARI | Best other | Winner |
|---------|-----------|------------|--------|
| **circles** | **1.000** | HDBSCAN 1.000 | tie — both topology-aware methods win; KMeans/GMM/Ward score ≈ 0 |
| **penguins** | 0.951 | GMM 0.959 | GMM by a whisker |
| **wine** | 0.568 | KMeans/GMM 0.897 | KMeans/GMM — 13-d Gaussian blobs hurt ToMATo |
| **digits 0–4** | 0.760 | Ward 0.857 | Ward — 64-d density estimate degrades |

### When to choose ToMATo
- Data has **non-convex topology** (rings, spirals, clusters with holes/handles).
- Dimensionality is **low (≤ ~10)**; k-NN density estimation degrades in high-d.
- You want a **single intuitive knob** (`k`) rather than the two in DBSCAN (eps + min_samples).

### When NOT to choose ToMATo
- High-dimensional data (digits, text embeddings) — density estimation breaks down.
- Gaussian/convex blobs (wine) — GMM and KMeans are faster and equally accurate.
- Very large datasets (> 50 k rows) — k-NN graph build time dominates.

### Comparison with other topology-adjacent methods
| Method | Topology mechanism | Limitation |
|--------|-------------------|------------|
| **ToMATo** | persistent-homology mode merging | high-d density failure |
| **HDBSCAN** | hierarchical density tree | can leave points as noise (-1) |
| **Spectral** | graph Laplacian eigenmaps | O(n²) memory; excluded here (slow on 64-d) |
